# 🏈 Análisis exploratorio de jugadores · props para parley

Exploración de `nflreadpy.load_player_stats()` (un renglón por **jugador × partido**) para
encontrar, por posición, a los jugadores más **confiables** para un parley.

La idea: un prop (p. ej. *"Mahomes más de 1.5 pases de TD"*) se gana si el jugador repite su
nivel. Así que no basta con un promedio alto — también buscamos **poca variabilidad**:

| Concepto | Qué mide | Qué queremos |
|---|---|---|
| `media` | Nivel típico por partido | Alto (≥ la línea) |
| `std` | Qué tanto se mueve partido a partido | **Bajo** |
| `cv` = std / media | Variabilidad relativa (compara jugadores de distinto nivel) | Bajo |
| `piso` = media − std | Un partido "malo pero normal" | Por encima de la línea |
| `hit_rate` | % de partidos en que superó la línea | Alto |
| `p_normal` | P(X > línea) suponiendo X ~ Normal(media, std) | Alto |

Diccionario de columnas: <https://nflreadr.nflverse.com/articles/dictionary_player_stats.html>

**Todo está en funciones**: cambia `TEMPORADAS` / `SEMANAS` en la celda de parámetros
(por default, la temporada actual completa) y vuelve a correr el notebook.

> ⚠️ Con pocas semanas jugadas la `std` es ruidosa (n pequeña). Usa `min_partidos` y
> toma los rankings como orientación, no como certeza.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import nflreadpy as nfl

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='notebook')

## 1 · Parámetros

- `TEMPORADAS`: `None` = temporada actual · `2025` · `[2024, 2025]` …
- `SEMANAS`: `None` = todas · `(4, 7)` = de la semana 4 a la 7 · `[5, 6, 7]`
- `ULTIMAS_N`: si se define, usa solo los últimos N partidos de cada jugador (forma reciente)

In [ ]:
TEMPORADAS   = None        # None -> temporada actual
SEMANAS      = None        # None -> todas las semanas disponibles
SEASON_TYPE  = 'REG'       # 'REG', 'POST' o None (ambas)
ULTIMAS_N    = None        # p. ej. 4 -> forma reciente
MIN_PARTIDOS = 3           # partidos mínimos para entrar al ranking
TOP_N        = 10          # jugadores a etiquetar / mostrar

## 2 · Métricas por posición

Para cada posición: las métricas que normalmente se usan como prop y un **filtro de volumen**
por partido, para no mezclar titulares con suplentes que entraron dos jugadas (eso infla la
`std` y ensucia las distribuciones).

Métricas derivadas que no vienen en el dataset:

- `scrimmage_yards` = `rushing_yards` + `receiving_yards`
- `total_tds` = `rushing_tds` + `receiving_tds`
- `touches` = `carries` + `receptions`
- `pass_rush_yards` = `passing_yards` + `rushing_yards` (QB)

In [ ]:
METRICAS_POSICION = {
    'QB': ['passing_yards', 'passing_tds', 'completions', 'attempts',
           'passing_interceptions', 'rushing_yards', 'pass_rush_yards', 'fantasy_points'],
    'RB': ['rushing_yards', 'carries', 'rushing_tds', 'receptions',
           'receiving_yards', 'scrimmage_yards', 'total_tds', 'fantasy_points_ppr'],
    'WR': ['receiving_yards', 'receptions', 'targets', 'receiving_tds',
           'receiving_air_yards', 'total_tds', 'fantasy_points_ppr'],
    'TE': ['receiving_yards', 'receptions', 'targets', 'receiving_tds',
           'total_tds', 'fantasy_points_ppr'],
}

# Métrica principal de cada posición (la que se usa en las vistas resumidas)
METRICA_PRINCIPAL = {'QB': 'passing_yards', 'RB': 'rushing_yards',
                     'WR': 'receiving_yards', 'TE': 'receiving_yards'}

# Volumen mínimo POR PARTIDO para que ese partido cuente (filtra apariciones marginales)
VOLUMEN_MINIMO = {
    'QB': ('attempts', 15),
    'RB': ('touches', 6),
    'WR': ('targets', 3),
    'TE': ('targets', 2),
}

NOMBRES = {
    'passing_yards': 'Yardas por pase', 'passing_tds': 'Pases de TD',
    'completions': 'Pases completos', 'attempts': 'Pases intentados',
    'passing_interceptions': 'Intercepciones', 'rushing_yards': 'Yardas por tierra',
    'pass_rush_yards': 'Yardas pase + tierra', 'fantasy_points': 'Fantasy points',
    'carries': 'Acarreos', 'rushing_tds': 'TD por tierra', 'receptions': 'Recepciones',
    'receiving_yards': 'Yardas por recepción', 'scrimmage_yards': 'Yardas totales (tierra + aire)',
    'total_tds': 'TD totales', 'fantasy_points_ppr': 'Fantasy points PPR',
    'targets': 'Targets', 'receiving_tds': 'TD por recepción',
    'receiving_air_yards': 'Air yards', 'touches': 'Toques',
}

def nombre(metrica):
    return NOMBRES.get(metrica, metrica)

## 3 · Carga y preparación

In [ ]:
def cargar_player_stats(temporadas=None, semanas=None, season_type='REG'):
    """
    Descarga las estadísticas por partido con nflreadpy y filtra por fechas.

    temporadas  : None (actual) | int | list[int]
    semanas     : None (todas) | (inicio, fin) inclusivo | list[int]
    season_type : 'REG' | 'POST' | None (ambas)
    """
    df = nfl.load_player_stats(seasons=temporadas, summary_level='week').to_pandas()

    if season_type is not None:
        df = df[df['season_type'] == season_type]
    if semanas is not None:
        if isinstance(semanas, tuple):
            df = df[df['week'].between(*semanas)]
        else:
            df = df[df['week'].isin(semanas)]

    return agregar_metricas_derivadas(df).reset_index(drop=True)


def agregar_metricas_derivadas(df):
    """Columnas combinadas que suelen usarse como prop."""
    df = df.copy()
    num = lambda c: df[c].fillna(0)
    df['scrimmage_yards'] = num('rushing_yards') + num('receiving_yards')
    df['total_tds'] = num('rushing_tds') + num('receiving_tds')
    df['touches'] = num('carries') + num('receptions')
    df['pass_rush_yards'] = num('passing_yards') + num('rushing_yards')
    return df


def datos_posicion(df, posicion, volumen_minimo=VOLUMEN_MINIMO, ultimas_n=None):
    """
    Renglones (jugador × partido) de una posición, solo partidos con volumen real.
    ultimas_n : si se da, conserva los últimos N partidos de cada jugador.
    """
    d = df[df['position'] == posicion].copy()
    if posicion in volumen_minimo:
        col, minimo = volumen_minimo[posicion]
        d = d[d[col].fillna(0) >= minimo]
    d = d.sort_values(['player_id', 'season', 'week'])
    if ultimas_n is not None:
        d = d.groupby('player_id', group_keys=False).tail(ultimas_n)
    metricas = METRICAS_POSICION.get(posicion, [])
    d[metricas] = d[metricas].fillna(0)
    return d.reset_index(drop=True)

In [ ]:
stats = cargar_player_stats(TEMPORADAS, SEMANAS, SEASON_TYPE)

print(f"Renglones (jugador × partido): {len(stats):,}")
print(f"Temporadas : {sorted(stats['season'].unique())}")
print(f"Semanas    : {sorted(stats['week'].unique())}")
print(f"Jugadores  : {stats['player_id'].nunique():,}")
stats.head()

### 3.1 · Panorama del dataset

Cuántos jugadores y partidos hay por posición, y cuántos sobreviven al filtro de volumen.

In [ ]:
def panorama_posiciones(df, posiciones=('QB', 'RB', 'WR', 'TE'), min_partidos=MIN_PARTIDOS):
    filas = []
    for pos in posiciones:
        bruto = df[df['position'] == pos]
        util = datos_posicion(df, pos)
        partidos = util.groupby('player_id').size()
        filas.append({
            'posicion': pos,
            'renglones': len(bruto),
            'jugadores': bruto['player_id'].nunique(),
            'renglones_con_volumen': len(util),
            'jugadores_con_volumen': util['player_id'].nunique(),
            f'jugadores_>=_{min_partidos}_partidos': int((partidos >= min_partidos).sum()),
        })
    return pd.DataFrame(filas).set_index('posicion')


def plot_panorama(panorama):
    cols = [c for c in panorama.columns if c.startswith('jugadores')]
    ax = panorama[cols].plot.bar(figsize=(10, 4), rot=0, width=0.8,
                                 color=['#c9cfe0', '#8e97b5', '#1e1b18'])
    ax.set_title('Jugadores por posición: total · con volumen · elegibles para el ranking')
    ax.set_xlabel('')
    for cont in ax.containers:
        ax.bar_label(cont, fontsize=8)
    plt.tight_layout()
    plt.show()


panorama = panorama_posiciones(stats)
display(panorama)
plot_panorama(panorama)

## 4 · Resumen por jugador (media, std, piso, hit rate)

`resumen_jugadores` colapsa los partidos de cada jugador en una fila. Si das una `linea`
(la línea del prop en la casa de apuestas) calcula además:

- `hit_rate`: % de partidos **por encima** de la línea (empírico)
- `p_normal`: P(X > línea) con una Normal(media, std) — más estable que el hit rate con pocas
  semanas, pero supone simetría (mala aproximación para TDs, que son conteos chicos)

In [ ]:
def resumen_jugadores(df_pos, metrica, min_partidos=MIN_PARTIDOS, linea=None):
    """Una fila por jugador con la distribución de `metrica` por partido."""
    g = df_pos.groupby(['player_id', 'player_display_name'])
    res = g[metrica].agg(partidos='count', media='mean', std='std', mediana='median',
                         minimo='min', maximo='max').reset_index()
    res['equipo'] = g['team'].last().values  # equipo más reciente
    res = res[res['partidos'] >= min_partidos].copy()

    res['std'] = res['std'].fillna(0)
    res['cv'] = np.where(res['media'] > 0, res['std'] / res['media'], np.nan)
    res['piso'] = res['media'] - res['std']

    if linea is not None:
        hits = df_pos.assign(_hit=df_pos[metrica] > linea).groupby('player_id')['_hit'].mean()
        res['hit_rate'] = res['player_id'].map(hits)
        std_seguro = res['std'].replace(0, 1e-9)
        res['p_normal'] = 1 - norm.cdf(linea, loc=res['media'], scale=std_seguro)

    cols = ['player_display_name', 'equipo', 'partidos', 'media', 'std', 'cv',
            'piso', 'mediana', 'minimo', 'maximo']
    cols += [c for c in ('hit_rate', 'p_normal') if c in res]
    return (res[cols].rename(columns={'player_display_name': 'jugador'})
            .sort_values('media', ascending=False).reset_index(drop=True))

## 5 · Gráficas

| Función | Qué responde |
|---|---|
| `scatter_media_std` | ¿Quién produce mucho **y** es estable? (cuadrante abajo-derecha) |
| `hist_metrica` | ¿Cómo se distribuye la métrica en la posición? ¿dónde caen las líneas típicas? |
| `barplot_top` | Top por media, con ±1 std como barra de error |
| `box_jugadores` | Distribución partido a partido de los mejores candidatos |
| `heatmap_semanas` | Consistencia semana a semana (detecta partidos atípicos / bye) |
| `corr_metricas` | Qué métricas se mueven juntas (evita meter props redundantes al parley) |

In [ ]:
def scatter_media_std(res, metrica, posicion, top_n=TOP_N, linea=None, ax=None):
    """
    Media vs std por jugador. Lo ideal: abajo a la derecha (mucha producción, poca variación).
    Las líneas punteadas son las medianas de la posición; el tamaño del punto = partidos.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 7))
    color = res['cv'].clip(upper=res['cv'].quantile(0.95))
    sc = ax.scatter(res['media'], res['std'], s=res['partidos'] * 25, c=color,
                    cmap='RdYlGn_r', alpha=0.8, edgecolor='k', linewidth=0.4)
    plt.colorbar(sc, ax=ax, label='CV (std / media)')

    x_med, y_med = res['media'].median(), res['std'].median()
    ax.axvline(x_med, ls='--', c='gray', lw=1)
    ax.axhline(y_med, ls='--', c='gray', lw=1)
    ax.text(ax.get_xlim()[1], ax.get_ylim()[0], 'alto y estable  ', ha='right',
            va='bottom', color='green', fontsize=10, weight='bold')
    if linea is not None:
        ax.axvline(linea, c='#b5523b', lw=1.5, label=f'línea = {linea}')
        ax.legend(loc='upper left')

    # Etiquetas: los mejores por media y los más estables entre los de media alta
    altos = res[res['media'] >= x_med]
    etiquetar = pd.concat([res.nlargest(top_n, 'media'), altos.nsmallest(top_n // 2, 'std')])
    for _, r in etiquetar.drop_duplicates('jugador').iterrows():
        ax.annotate(r['jugador'], (r['media'], r['std']), fontsize=8,
                    xytext=(4, 4), textcoords='offset points')

    ax.set_xlabel(f'Media por partido · {nombre(metrica)}')
    ax.set_ylabel('Desviación estándar')
    ax.set_title(f'{posicion} · {nombre(metrica)}: media vs. variabilidad')
    return ax


def hist_metrica(df_pos, metrica, posicion, lineas=None, ax=None):
    """Histograma + KDE de la métrica por partido (todos los jugadores de la posición)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    valores = df_pos[metrica]
    discreta = valores.nunique() <= 8   # TDs, INTs: barras por valor entero
    sns.histplot(valores, ax=ax, discrete=discreta, kde=not discreta,
                 color='#8e97b5', edgecolor='white')
    ax.axvline(valores.mean(), c='k', lw=1.5, label=f'media = {valores.mean():.1f}')
    ax.axvline(valores.median(), c='k', ls='--', lw=1, label=f'mediana = {valores.median():.1f}')
    for l in (lineas or []):
        pct = (valores > l).mean()
        ax.axvline(l, c='#b5523b', lw=1.2, label=f'> {l}: {pct:.0%} de los partidos')
    ax.set_title(f'{posicion} · distribución de {nombre(metrica)} por partido')
    ax.set_xlabel(nombre(metrica))
    ax.legend(fontsize=8)
    return ax


def barplot_top(res, metrica, posicion, top_n=TOP_N, orden='media', ax=None):
    """Top N por `orden` (media, piso, p_normal…) con la media ± 1 std."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 0.45 * top_n + 1.5))
    top = res.nlargest(top_n, orden).iloc[::-1]
    etiquetas = top['jugador'] + ' (' + top['equipo'] + ')'
    ax.barh(etiquetas, top['media'], xerr=top['std'], color='#7a6a50', alpha=0.85,
            error_kw={'ecolor': '#1e1b18', 'capsize': 3, 'lw': 1})
    ax.scatter(top['piso'].clip(lower=0), etiquetas, marker='|', s=200, c='#b5523b',
               zorder=3, label='piso (media − std)')
    ax.set_xlabel(f'{nombre(metrica)} por partido (barra = media · línea = ±1 std)')
    ax.set_title(f'{posicion} · top {top_n} por {orden}')
    ax.legend(loc='lower right', fontsize=8)
    return ax


def box_jugadores(df_pos, metrica, jugadores, posicion, linea=None, ax=None):
    """Distribución partido a partido de una lista de jugadores (box + puntos)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 0.5 * len(jugadores) + 1.5))
    d = df_pos[df_pos['player_display_name'].isin(jugadores)]
    orden = list(jugadores)
    sns.boxplot(data=d, y='player_display_name', x=metrica, order=orden, ax=ax,
                color='#e2d8cb', fliersize=0)
    sns.stripplot(data=d, y='player_display_name', x=metrica, order=orden, ax=ax,
                  color='#1e1b18', size=5, alpha=0.7)
    if linea is not None:
        ax.axvline(linea, c='#b5523b', lw=1.5, label=f'línea = {linea}')
        ax.legend(fontsize=8)
    ax.set_ylabel('')
    ax.set_xlabel(nombre(metrica))
    ax.set_title(f'{posicion} · {nombre(metrica)} partido a partido')
    return ax


def heatmap_semanas(df_pos, metrica, jugadores, posicion, ax=None):
    """Jugador × semana. Celdas vacías = no jugó o no llegó al volumen mínimo."""
    d = df_pos[df_pos['player_display_name'].isin(jugadores)]
    tabla = d.pivot_table(index='player_display_name', columns='week', values=metrica,
                          aggfunc='sum').reindex(jugadores)
    if ax is None:
        _, ax = plt.subplots(figsize=(1.0 * tabla.shape[1] + 3, 0.45 * len(jugadores) + 1.5))
    sns.heatmap(tabla, annot=True, fmt='.0f', cmap='YlGnBu', cbar=False, ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_ylabel('')
    ax.set_xlabel('Semana')
    ax.set_title(f'{posicion} · {nombre(metrica)} por semana')
    return ax


def corr_metricas(df_pos, posicion, metricas=None, ax=None):
    """Correlación entre métricas por partido (Spearman, robusta a colas largas)."""
    metricas = metricas or METRICAS_POSICION[posicion]
    if ax is None:
        _, ax = plt.subplots(figsize=(0.8 * len(metricas) + 2, 0.7 * len(metricas) + 1.5))
    corr = df_pos[metricas].corr(method='spearman')
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, ax=ax,
                mask=np.triu(np.ones_like(corr, dtype=bool), k=1),
                xticklabels=[nombre(m) for m in metricas],
                yticklabels=[nombre(m) for m in metricas])
    ax.set_title(f'{posicion} · correlación entre métricas (Spearman)')
    return ax

## 6 · Ranking para parley

`mejores_para_parley` implementa la regla *"quiero un jugador con media ≥ x y, entre esos,
la menor std"*:

1. Filtra `media >= media_min` (si no se da, usa la `linea`; si tampoco, la mediana de la posición)
2. Opcional: `std_max` para descartar a los volátiles
3. Ordena por `orden` — por default `std` ascendente. Alternativas útiles:
   `'cv'` (variabilidad relativa), `'-piso'` (mayor piso), `'-p_normal'`, `'-hit_rate'`

`prob_parley` multiplica las probabilidades de las piernas elegidas (supone independencia:
evita juntar props del mismo partido o muy correlacionados — ver `corr_metricas`).

In [ ]:
def mejores_para_parley(df_pos, metrica, media_min=None, std_max=None, linea=None,
                        orden='std', min_partidos=MIN_PARTIDOS, top_n=TOP_N):
    """
    Candidatos para un prop de `metrica`: media alta y la menor variabilidad.

    orden : columna para ordenar; prefijo '-' = descendente ('std', 'cv', '-piso', '-p_normal')
    """
    res = resumen_jugadores(df_pos, metrica, min_partidos=min_partidos, linea=linea)
    if media_min is None:
        media_min = linea if linea is not None else res['media'].median()
    res = res[res['media'] >= media_min]
    if std_max is not None:
        res = res[res['std'] <= std_max]

    desc = orden.startswith('-')
    col = orden.lstrip('-')
    return res.sort_values(col, ascending=not desc).head(top_n).reset_index(drop=True)


def prob_parley(*probabilidades):
    """Probabilidad conjunta de un parley suponiendo piernas independientes."""
    p = float(np.prod(probabilidades))
    return {'prob': p, 'momio_justo_decimal': 1 / p if p > 0 else np.inf}


def tabla_estilizada(res):
    """Formato rápido para mostrar resúmenes en el notebook."""
    fmt = {c: '{:.1f}' for c in ('media', 'std', 'piso', 'mediana', 'minimo', 'maximo')}
    fmt.update({'cv': '{:.2f}', 'hit_rate': '{:.0%}', 'p_normal': '{:.0%}'})
    fmt = {k: v for k, v in fmt.items() if k in res}
    sty = res.style.format(fmt).background_gradient(subset=['std'], cmap='RdYlGn_r')
    if 'p_normal' in res:
        sty = sty.background_gradient(subset=['p_normal'], cmap='RdYlGn', vmin=0, vmax=1)
    return sty

## 7 · Vista por posición

`vista_posicion` junta todo para una posición: panel de scatter media-vs-std para cada métrica,
histogramas, top por media, distribución de los candidatos, heatmap semanal, correlación y la
tabla de candidatos para parley de la métrica principal.

In [ ]:
def panel_scatter(df_pos, posicion, metricas=None, min_partidos=MIN_PARTIDOS, top_n=5):
    """Un scatter media-vs-std por métrica, en cuadrícula."""
    metricas = metricas or METRICAS_POSICION[posicion]
    ncols = 2
    nrows = int(np.ceil(len(metricas) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 6 * nrows))
    for ax, m in zip(axes.flat, metricas):
        res = resumen_jugadores(df_pos, m, min_partidos)
        scatter_media_std(res, m, posicion, top_n=top_n, ax=ax)
    for ax in list(axes.flat)[len(metricas):]:
        ax.remove()
    fig.tight_layout()
    plt.show()


def panel_histogramas(df_pos, posicion, metricas=None, lineas=None):
    """Histogramas de todas las métricas de la posición. lineas = {metrica: [l1, l2]}"""
    metricas = metricas or METRICAS_POSICION[posicion]
    lineas = lineas or {}
    ncols = 3
    nrows = int(np.ceil(len(metricas) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
    for ax, m in zip(axes.flat, metricas):
        hist_metrica(df_pos, m, posicion, lineas=lineas.get(m), ax=ax)
    for ax in list(axes.flat)[len(metricas):]:
        ax.remove()
    fig.tight_layout()
    plt.show()


def vista_posicion(df, posicion, metrica=None, linea=None, media_min=None,
                   min_partidos=MIN_PARTIDOS, top_n=TOP_N, ultimas_n=ULTIMAS_N,
                   lineas_hist=None):
    """
    EDA completa de una posición. Devuelve (df_pos, candidatos) para seguir explorando.

    metrica : métrica para el ranking / box / heatmap (default: METRICA_PRINCIPAL)
    linea   : línea del prop; activa hit_rate / p_normal y se dibuja en las gráficas
    """
    metrica = metrica or METRICA_PRINCIPAL[posicion]
    df_pos = datos_posicion(df, posicion, ultimas_n=ultimas_n)
    print(f"{posicion}: {df_pos['player_id'].nunique()} jugadores · {len(df_pos)} partidos con volumen")

    panel_scatter(df_pos, posicion, min_partidos=min_partidos)
    panel_histogramas(df_pos, posicion, lineas=lineas_hist or ({metrica: [linea]} if linea else None))

    res = resumen_jugadores(df_pos, metrica, min_partidos, linea=linea)
    fig, axes = plt.subplots(1, 2, figsize=(18, 0.45 * top_n + 2))
    barplot_top(res, metrica, posicion, top_n=top_n, orden='media', ax=axes[0])
    barplot_top(res, metrica, posicion, top_n=top_n, orden='piso', ax=axes[1])
    fig.tight_layout()
    plt.show()

    candidatos = mejores_para_parley(df_pos, metrica, media_min=media_min, linea=linea,
                                     min_partidos=min_partidos, top_n=top_n)
    if not candidatos.empty:
        jugadores = candidatos['jugador'].tolist()
        box_jugadores(df_pos, metrica, jugadores, posicion, linea=linea)
        plt.show()
        heatmap_semanas(df_pos, metrica, jugadores, posicion)
        plt.show()

    corr_metricas(df_pos, posicion)
    plt.show()

    print(f"Candidatos para parley · {nombre(metrica)} · media ≥ "
          f"{media_min if media_min is not None else (linea if linea is not None else 'mediana')}"
          f" · ordenados por menor std")
    display(tabla_estilizada(candidatos))
    return df_pos, candidatos

### 7.1 · QB · Quarterbacks

Métrica principal: **yardas por pase**. Línea de ejemplo 224.5 (rango típico de props de QB titular). Prueba también `metrica='passing_tds', linea=1.5`.

In [ ]:
df_qb, cand_qb = vista_posicion(stats, 'QB', metrica='passing_yards', linea=224.5)

### 7.2 · RB · Running backs

Métrica principal: **yardas por tierra**. Línea de ejemplo 59.5. Para RBs de doble función, `scrimmage_yards` suele ser más estable que solo tierra.

In [ ]:
df_rb, cand_rb = vista_posicion(stats, 'RB', metrica='rushing_yards', linea=59.5)

### 7.3 · WR · Wide receivers

Métrica principal: **yardas por recepción**. Línea de ejemplo 54.5. `receptions` suele tener menor CV que las yardas (buen prop "seguro").

In [ ]:
df_wr, cand_wr = vista_posicion(stats, 'WR', metrica='receiving_yards', linea=54.5)

### 7.4 · TE · Tight ends

Métrica principal: **yardas por recepción**. Línea de ejemplo 39.5.

In [ ]:
df_te, cand_te = vista_posicion(stats, 'TE', metrica='receiving_yards', linea=39.5)

## 8 · Otras métricas y forma reciente

Las funciones se pueden usar sueltas. Ejemplos:

In [ ]:
# QB · pases de TD con línea 1.5, ordenando por probabilidad (Normal) de superar la línea
tabla_estilizada(mejores_para_parley(df_qb, 'passing_tds', linea=1.5, orden='-p_normal'))

In [ ]:
# WR · recepciones: media ≥ 5 y std ≤ 2, ordenados por CV
tabla_estilizada(mejores_para_parley(df_wr, 'receptions', media_min=5, std_max=2, orden='cv'))

In [ ]:
# Forma reciente: RB con solo los últimos 3 partidos de cada jugador
df_rb_reciente = datos_posicion(stats, 'RB', ultimas_n=3)
res = resumen_jugadores(df_rb_reciente, 'scrimmage_yards', min_partidos=3)
scatter_media_std(res, 'scrimmage_yards', 'RB (últimos 3)')
plt.show()

In [ ]:
# Otra ventana de fechas: temporada pasada completa (para comparar estabilidad año contra año)
# stats_2025 = cargar_player_stats(2025)
# vista_posicion(stats_2025, 'QB', linea=224.5)

## 9 · Armar el parley

De cada tabla de candidatos (media ≥ línea, los de menor std) toma al de mayor `p_normal`
y calcula la probabilidad conjunta. Si prefieres lo observado, usa `columna='hit_rate'`.

In [ ]:
def armar_parley(*candidatos, columna='p_normal'):
    """
    De cada tabla de candidatos (ya filtrada por media y ordenada por std) toma al de mayor
    `columna` y calcula la probabilidad del parley.
    """
    piernas = pd.DataFrame([c.loc[c[columna].idxmax()] for c in candidatos if not c.empty])
    resultado = prob_parley(*piernas[columna])
    display(piernas[['jugador', 'equipo', 'media', 'std', 'piso', 'hit_rate', 'p_normal']])
    print(f"Probabilidad del parley ({columna}, piernas independientes): {resultado['prob']:.1%}"
          f" · momio justo decimal: {resultado['momio_justo_decimal']:.2f}")
    return piernas, resultado


piernas, resultado = armar_parley(cand_qb, cand_rb, cand_wr, cand_te)

## 10 · Vista por partido (dos equipos)

Para armar un parley de **un partido**: eliges dos equipos y ves quién anota TD, quién recibe
el volumen y cómo llegan los equipos en puntos.

| Función | Qué devuelve |
|---|---|
| `cargar_calendario(temporadas)` | Calendario con marcadores y líneas de Vegas (spread, total, moneyline) |
| `partido_del_dia(cal, fecha)` | Partidos de una fecha (default: hoy) |
| `marcadores_equipo(cal, equipo)` | Puntos a favor / en contra y total de cada partido, vs. la línea de total |
| `resumen_equipo(stats, equipo, previa)` | Una fila por jugador: volumen, yardas, TDs, % de partidos con TD y `p_td` |
| `td_permitidos(stats, equipo)` | TDs que permite la defensa por posición vs. el promedio de la liga |
| `vista_partido(stats, 'CHI', 'SEA')` | Todo lo anterior + gráficas |

**`p_td`** (probabilidad de anotar al menos un TD por tierra o recepción) usa un modelo de
Poisson: λ = TDs por partido, con la temporada previa a `peso_previa` (default ½) para
estabilizar muestras chicas → `p_td = 1 − e^(−λ)`. Se reporta junto con el `hit_td`
observado (% de partidos con TD) para que veas ambos.

> ⚠️ En un parley del **mismo partido** las piernas están correlacionadas (si un equipo anota
> mucho, suben el over, sus TDs y sus yardas a la vez). No multipliques probabilidades como
> si fueran independientes.

In [ ]:
from pathlib import Path

POSICIONES_OFENSIVAS = ['QB', 'RB', 'WR', 'TE']
COLORES_EQUIPO = ['#7a6a50', '#8e97b5']   # equipo A (visitante), equipo B (local)


def cargar_calendario(temporadas=None):
    """Calendario con marcadores y líneas. None -> temporada actual."""
    if temporadas is None:
        temporadas = [nfl.get_current_season()]
    return nfl.load_schedules(seasons=temporadas).to_pandas()


def partido_del_dia(cal, fecha=None):
    """Partidos de `fecha` ('YYYY-MM-DD'; default hoy) con líneas de Vegas."""
    fecha = fecha or pd.Timestamp.today().strftime('%Y-%m-%d')
    cols = ['week', 'gameday', 'gametime', 'away_team', 'home_team', 'spread_line',
            'total_line', 'away_moneyline', 'home_moneyline', 'away_qb_name', 'home_qb_name']
    return cal.loc[cal['gameday'] == fecha, cols].reset_index(drop=True)


def marcadores_equipo(cal, equipo, season_type='REG'):
    """Un renglón por partido jugado: puntos a favor, en contra, total y si pasó la línea."""
    c = cal[(cal['home_score'].notna()) & ((cal['home_team'] == equipo) | (cal['away_team'] == equipo))]
    if season_type is not None:
        c = c[c['game_type'] == season_type]
    local = c['home_team'] == equipo
    m = pd.DataFrame({
        'semana': c['week'],
        'rival': np.where(local, c['away_team'], '@' + c['home_team']),
        'pf': np.where(local, c['home_score'], c['away_score']),
        'pc': np.where(local, c['away_score'], c['home_score']),
        'linea_total': c['total_line'],
    })
    m['total'] = m['pf'] + m['pc']
    m['gano'] = m['pf'] > m['pc']
    m['over'] = m['total'] > m['linea_total']
    return m.reset_index(drop=True)


def resumen_marcadores(cal, equipos):
    """Promedios de puntos por equipo (una fila por equipo)."""
    filas = []
    for eq in equipos:
        m = marcadores_equipo(cal, eq)
        filas.append({
            'equipo': eq, 'partidos': len(m),
            'record': f"{int(m['gano'].sum())}-{int((~m['gano']).sum())}",
            'pf_media': m['pf'].mean(), 'pf_std': m['pf'].std(),
            'pc_media': m['pc'].mean(), 'pc_std': m['pc'].std(),
            'total_media': m['total'].mean(), 'total_mediana': m['total'].median(),
            'total_std': m['total'].std(),
            'overs': f"{int(m['over'].sum())}/{int(m['linea_total'].notna().sum())}",
        })
    return pd.DataFrame(filas).set_index('equipo')


def proyeccion_total(cal, equipo_a, equipo_b, linea=None):
    """
    Puntos esperados de cada equipo = promedio de (lo que anota) y (lo que permite el rival).
    Con `linea`, P(over) suponiendo total ~ Normal(proyección, std combinada de sus totales).
    """
    ma, mb = marcadores_equipo(cal, equipo_a), marcadores_equipo(cal, equipo_b)
    pts_a = (ma['pf'].mean() + mb['pc'].mean()) / 2
    pts_b = (mb['pf'].mean() + ma['pc'].mean()) / 2
    pts_a_med = (ma['pf'].median() + mb['pc'].median()) / 2
    pts_b_med = (mb['pf'].median() + ma['pc'].median()) / 2
    out = {f'pts_{equipo_a}': pts_a, f'pts_{equipo_b}': pts_b, 'total_proyectado': pts_a + pts_b,
           'total_proyectado_medianas': pts_a_med + pts_b_med}
    if linea is not None:
        std = pd.concat([ma['total'], mb['total']]).std()
        out['linea'] = linea
        out['p_over'] = 1 - norm.cdf(linea, loc=pts_a + pts_b, scale=std)
        out['overs_historicos'] = f"{int((ma['total'] > linea).sum() + (mb['total'] > linea).sum())}/{len(ma) + len(mb)}"
    return out


def prob_elo(equipo_local, equipo_visitante, semana, temporada=None,
             data_dir=Path('..') / 'data' / 'processed'):
    """Probabilidad del ELO in-season (Notebook 05 / weekly_content.py), si el CSV existe."""
    temporada = temporada or nfl.get_current_season()
    ruta = data_dir / f'weekly_predictions_{temporada}.csv'
    if not ruta.exists():
        return None
    w = pd.read_csv(ruta)
    fila = w[(w['week'] == semana) & (w['home_team'] == equipo_local) & (w['away_team'] == equipo_visitante)]
    if fila.empty:
        return None
    f = fila.iloc[0]
    return {equipo_local: f['p_home_pre'], equipo_visitante: f['p_away_pre']}


def prob_moneyline(ml):
    """Momio americano -> probabilidad implícita (con margen de la casa)."""
    return 100 / (ml + 100) if ml > 0 else -ml / (-ml + 100)

In [ ]:
def resumen_equipo(df, equipo, previa=None, peso_previa=0.5, min_volumen=2):
    """
    Una fila por jugador ofensivo del equipo.

    df          : stats por partido (cargar_player_stats)
    previa      : stats de la temporada anterior (cualquier equipo) para estabilizar p_td
    min_volumen : toques + targets mínimos por partido (en promedio) para aparecer
    """
    d = df[(df['team'] == equipo) & df['position'].isin(POSICIONES_OFENSIVAS)].copy()
    d['td'] = d['rushing_tds'].fillna(0) + d['receiving_tds'].fillna(0)
    d['volumen'] = d['carries'].fillna(0) + d['targets'].fillna(0)
    ultima = d['week'].max()

    g = d.groupby('player_id')
    res = g.agg(jugador=('player_display_name', 'last'), pos=('position', 'last'),
                partidos=('week', 'nunique'), ultima_semana=('week', 'max'),
                acarreos=('carries', 'mean'), targets=('targets', 'mean'),
                volumen=('volumen', 'mean'),
                yardas_tierra=('rushing_yards', 'mean'), yardas_recepcion=('receiving_yards', 'mean'),
                yardas_totales=('scrimmage_yards', 'mean'), std_yardas_totales=('scrimmage_yards', 'std'),
                pases_td=('passing_tds', 'mean'), tds=('td', 'sum'),
                partidos_con_td=('td', lambda x: int((x >= 1).sum())))
    res = res[(res['volumen'] >= min_volumen) | (res['pos'] == 'QB')]
    res['jugo_ultima'] = res['ultima_semana'] == ultima
    res['hit_td'] = res['partidos_con_td'] / res['partidos']

    # Participación en el volumen del equipo (toques + targets)
    vol_equipo = d.groupby('week')['volumen'].sum()
    vol_jugador = d.pivot_table(index='player_id', columns='week', values='volumen', aggfunc='sum')
    res['share_volumen'] = (vol_jugador.div(vol_equipo, axis=1).mean(axis=1)).reindex(res.index)

    # p_td con Poisson + temporada previa a medio peso
    tds_prev = pj_prev = 0
    if previa is not None:
        p = previa[previa['position'].isin(POSICIONES_OFENSIVAS)].copy()
        p['td'] = p['rushing_tds'].fillna(0) + p['receiving_tds'].fillna(0)
        gp = p.groupby('player_id').agg(pj=('week', 'nunique'), tds=('td', 'sum'))
        tds_prev = gp['tds'].reindex(res.index).fillna(0)
        pj_prev = gp['pj'].reindex(res.index).fillna(0)
        res['tds_previa'] = tds_prev
        res['partidos_previa'] = pj_prev
    lam = (res['tds'] + peso_previa * tds_prev) / (res['partidos'] + peso_previa * pj_prev)
    res['p_td'] = 1 - np.exp(-lam)

    res['equipo'] = equipo
    return res.sort_values('p_td', ascending=False).reset_index()


def td_permitidos(df, equipo, season_type='REG'):
    """TDs (tierra + recepción) por partido que permite la defensa de `equipo`, por posición."""
    d = df[df['position'].isin(POSICIONES_OFENSIVAS)].copy()
    if season_type is not None:
        d = d[d['season_type'] == season_type]
    d['td'] = d['rushing_tds'].fillna(0) + d['receiving_tds'].fillna(0)
    d['yardas'] = d['scrimmage_yards']

    def por_partido(sub):
        partidos = sub['game_id'].nunique()
        return sub.groupby('position')[['td', 'yardas']].sum() / partidos

    rival = por_partido(d[d['opponent_team'] == equipo])
    # Promedio de la liga: lo que permite una defensa típica por partido
    liga = d.groupby(['opponent_team', 'position'])[['td', 'yardas']].sum()
    partidos_def = d.groupby('opponent_team')['game_id'].nunique()
    liga = liga.div(partidos_def, axis=0, level='opponent_team').groupby('position').mean()

    t = pd.DataFrame({
        'td_permitidos': rival['td'], 'td_liga': liga['td'],
        'yardas_permitidas': rival['yardas'], 'yardas_liga': liga['yardas'],
    }).reindex(POSICIONES_OFENSIVAS)
    t['td_vs_liga'] = t['td_permitidos'] / t['td_liga']
    t['yardas_vs_liga'] = t['yardas_permitidas'] / t['yardas_liga']
    return t

In [ ]:
def plot_td_partido(resumenes, top_n=8, ax=None):
    """Ranking de p_td de los jugadores de ambos equipos, con el % observado de partidos con TD."""
    r = pd.concat(resumenes)
    r = r[r['pos'] != 'QB'].nlargest(top_n * 2, 'p_td').iloc[::-1]
    equipos = list(dict.fromkeys(x['equipo'].iloc[0] for x in resumenes))
    colores = r['equipo'].map(dict(zip(equipos, COLORES_EQUIPO)))
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 0.4 * len(r) + 1.5))
    etiquetas = r['jugador'] + ' · ' + r['pos'] + ' (' + r['equipo'] + ')'
    ax.barh(etiquetas, r['p_td'], color=colores, alpha=0.9)
    ax.scatter(r['hit_td'], etiquetas, marker='D', c='#1e1b18', s=30, zorder=3,
               label='% partidos con TD (observado)')
    for y, (p, n, k) in enumerate(zip(r['p_td'], r['partidos'], r['partidos_con_td'])):
        ax.text(p + 0.01, y, f'{p:.0%}  ({k}/{n})', va='center', fontsize=8)
    for eq, col in zip(equipos, COLORES_EQUIPO):
        ax.barh([], [], color=col, label=eq)
    ax.set_xlim(0, 1.05)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_xlabel('P(anota al menos 1 TD) · Poisson')
    ax.set_title('¿Quién anota TD? · ' + ' vs. '.join(equipos))
    ax.legend(loc='lower right', fontsize=8)
    return ax


def heatmap_td(df, equipo, jugadores, ax=None):
    """TDs (tierra + recepción) por semana de cada jugador. Vacío = no jugó."""
    d = df[(df['team'] == equipo) & df['player_display_name'].isin(jugadores)].copy()
    d['td'] = d['rushing_tds'].fillna(0) + d['receiving_tds'].fillna(0)
    t = d.pivot_table(index='player_display_name', columns='week', values='td', aggfunc='sum')
    t = t.reindex([j for j in jugadores if j in t.index])
    if ax is None:
        _, ax = plt.subplots(figsize=(0.8 * t.shape[1] + 3, 0.45 * len(t) + 1.5))
    sns.heatmap(t, annot=True, fmt='.0f', cmap='Greens', vmin=0, vmax=2, cbar=False,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_ylabel('')
    ax.set_xlabel('Semana')
    ax.set_title(f'{equipo} · TDs por semana')
    return ax


def plot_uso_semanal(df, equipo, top_n=6, ax=None):
    """Toques + targets por semana de los jugadores con más volumen: detecta cambios de rol."""
    d = df[(df['team'] == equipo) & df['position'].isin(['RB', 'WR', 'TE'])].copy()
    d['volumen'] = d['carries'].fillna(0) + d['targets'].fillna(0)
    top = d.groupby('player_display_name')['volumen'].sum().nlargest(top_n).index
    t = d[d['player_display_name'].isin(top)].pivot_table(
        index='week', columns='player_display_name', values='volumen', aggfunc='sum')
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    t[list(top)].plot(ax=ax, marker='o')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Acarreos + targets')
    ax.set_title(f'{equipo} · uso semanal (top {top_n})')
    ax.legend(fontsize=8, ncol=2)
    return ax


def plot_puntos(cal, equipos, linea_total=None, axes=None):
    """Puntos a favor / en contra por semana y total de cada partido vs. su línea."""
    if axes is None:
        _, axes = plt.subplots(1, len(equipos), figsize=(8 * len(equipos), 4), sharey=True)
    for ax, eq, col in zip(np.atleast_1d(axes), equipos, COLORES_EQUIPO):
        m = marcadores_equipo(cal, eq)
        x = np.arange(len(m))
        ax.bar(x - 0.2, m['pf'], 0.4, color=col, label='a favor')
        ax.bar(x + 0.2, m['pc'], 0.4, color='#c9cfe0', label='en contra')
        ax.plot(x, m['total'], 'o-', c='#1e1b18', label='total')
        ax.plot(x, m['linea_total'], '_', c='#b5523b', ms=18, mew=2, label='línea del total')
        if linea_total is not None:
            ax.axhline(linea_total, ls='--', c='#b5523b', lw=1, label=f'línea hoy = {linea_total}')
        ax.set_xticks(x, [f"S{s}\n{r}" for s, r in zip(m['semana'], m['rival'])], fontsize=8)
        ax.set_title(f'{eq} · puntos por partido')
        ax.legend(fontsize=7)
    return axes

In [ ]:
def tabla_equipo(res):
    """Vista compacta de resumen_equipo."""
    cols = ['jugador', 'pos', 'partidos', 'jugo_ultima', 'share_volumen', 'acarreos', 'targets',
            'yardas_tierra', 'yardas_recepcion', 'yardas_totales', 'std_yardas_totales',
            'pases_td', 'tds', 'partidos_con_td', 'hit_td', 'p_td']
    fmt = {c: '{:.1f}' for c in ('acarreos', 'targets', 'yardas_tierra', 'yardas_recepcion',
                                 'yardas_totales', 'std_yardas_totales', 'pases_td')}
    fmt.update({'share_volumen': '{:.0%}', 'hit_td': '{:.0%}', 'p_td': '{:.0%}'})
    return (res[cols].style.format(fmt, na_rep='—')
            .background_gradient(subset=['p_td'], cmap='Greens', vmin=0, vmax=0.8)
            .background_gradient(subset=['share_volumen'], cmap='Blues', vmin=0, vmax=0.4))


def vista_partido(df, equipo_a=None, equipo_b=None, cal=None, previa=None, fecha=None,
                  peso_previa=0.5, top_n=8):
    """
    Vista de un partido para armar parley.

    equipo_a, equipo_b : visitante y local. Si no se dan, usa el primer partido de `fecha` (hoy).
    previa             : stats de la temporada anterior para estabilizar p_td
    Devuelve un dict con las tablas para seguir explorando.
    """
    cal = cal if cal is not None else cargar_calendario(sorted(df['season'].unique()))
    juego = None
    if equipo_a is None or equipo_b is None:
        hoy = partido_del_dia(cal, fecha)
        if hoy.empty:
            raise ValueError(f'No hay partidos el {fecha or "día de hoy"}; pasa equipo_a y equipo_b')
        juego = hoy.iloc[0]
        equipo_a, equipo_b = juego['away_team'], juego['home_team']
    else:
        prox = cal[(cal['away_team'] == equipo_a) & (cal['home_team'] == equipo_b)]
        prox = prox[prox['home_score'].isna()]
        juego = prox.iloc[0] if not prox.empty else None

    print(f"══ {equipo_a} @ {equipo_b} ══")
    linea_total = None
    if juego is not None:
        linea_total = juego['total_line']
        print(f"Semana {juego['week']} · {juego['gameday']} {juego['gametime']} ET · "
              f"spread {equipo_b} -{juego['spread_line']} · total {linea_total} · "
              f"ML {equipo_a} {juego['away_moneyline']:+.0f} / {equipo_b} {juego['home_moneyline']:+.0f}")
        p_a, p_b = prob_moneyline(juego['away_moneyline']), prob_moneyline(juego['home_moneyline'])
        print(f"Vegas sin margen : {equipo_a} {p_a / (p_a + p_b):.1%} · {equipo_b} {p_b / (p_a + p_b):.1%}")
        elo = prob_elo(equipo_b, equipo_a, juego['week'], juego.get('season'))
        if elo:
            print(f"ELO (Notebook 05): {equipo_a} {elo[equipo_a]:.1%} · {equipo_b} {elo[equipo_b]:.1%}")

    # Equipos: puntos y total
    display(resumen_marcadores(cal, [equipo_a, equipo_b]).style.format(precision=1))
    proy = proyeccion_total(cal, equipo_a, equipo_b, linea=linea_total)
    print('Proyección: ' + ' · '.join(
        f'{k} = {v:.1%}' if k == 'p_over' else f'{k} = {v:.1f}' if isinstance(v, float) else f'{k} = {v}'
        for k, v in proy.items()))
    plot_puntos(cal, [equipo_a, equipo_b], linea_total=linea_total)
    plt.tight_layout()
    plt.show()

    # Jugadores
    resumenes = [resumen_equipo(df, eq, previa=previa, peso_previa=peso_previa)
                 for eq in (equipo_a, equipo_b)]
    plot_td_partido(resumenes, top_n=top_n)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    for ax, eq in zip(axes, (equipo_a, equipo_b)):
        plot_uso_semanal(df, eq, ax=ax)
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(18, 0.45 * top_n + 2))
    for ax, res in zip(axes, resumenes):
        jugadores = res[res['pos'] != 'QB'].head(top_n)['jugador'].tolist()
        heatmap_td(df, res['equipo'].iloc[0], jugadores, ax=ax)
    fig.tight_layout()
    plt.show()

    for res in resumenes:
        print(f"\n{res['equipo'].iloc[0]} · jugadores (ordenados por p_td)")
        display(tabla_equipo(res))

    # Defensas: qué permite cada una (lo que enfrentará el otro equipo)
    defensas = {}
    for ataque, defensa in ((equipo_a, equipo_b), (equipo_b, equipo_a)):
        defensas[defensa] = td_permitidos(df, defensa)
        print(f"\nDefensa de {defensa} (lo que enfrenta {ataque}) · por partido vs. promedio liga")
        display(defensas[defensa].style.format('{:.2f}')
                .background_gradient(subset=['td_vs_liga', 'yardas_vs_liga'], cmap='RdYlGn',
                                     vmin=0.5, vmax=1.5))

    return {'equipos': (equipo_a, equipo_b), 'juego': juego, 'proyeccion': proy,
            'jugadores': dict(zip((equipo_a, equipo_b), resumenes)), 'defensas': defensas}

### 10.1 · Partido de hoy · CHI @ SEA

`previa` = temporada anterior completa, para que `p_td` no dependa solo de 7–8 partidos.
Para otro partido: `vista_partido(stats, 'KC', 'BUF', previa=stats_previa)`.
Sin argumentos de equipos toma el primer partido del día (`fecha='YYYY-MM-DD'` para otro día).

In [ ]:
calendario = cargar_calendario(sorted(stats['season'].unique()))
stats_previa = cargar_player_stats(int(stats['season'].max()) - 1, season_type='REG')

display(partido_del_dia(calendario))
partido = vista_partido(stats, 'CHI', 'SEA', cal=calendario, previa=stats_previa)

### 10.2 · Del resumen al parley

Las tablas quedan en `partido['jugadores'][EQUIPO]`. Por ejemplo, para comparar `p_td` contra
el momio de la casa (`prob_moneyline(-150)` = probabilidad implícita):

In [ ]:
td = pd.concat(partido['jugadores'].values())
td = td[td['pos'] != 'QB'][['jugador', 'equipo', 'pos', 'partidos', 'partidos_con_td', 'hit_td', 'p_td']]

# Pon aquí los momios de "anota TD" de tu casa de apuestas para ver dónde hay valor
momios_td = {}   # p. ej. {'Kenneth Walker III': -150, "D'Andre Swift": +110}
td['momio'] = td['jugador'].map(momios_td)
td['p_casa'] = td['momio'].dropna().map(prob_moneyline)
td['ventaja'] = td['p_td'] - td['p_casa']
td.head(12).style.format({'hit_td': '{:.0%}', 'p_td': '{:.0%}', 'p_casa': '{:.0%}',
                          'ventaja': '{:+.0%}', 'momio': '{:+.0f}'}, na_rep='—')